# Political Season Analysis: Intraday Momentum Breakout Strategy

This notebook analyzes the performance of an intraday momentum breakout strategy across four distinct political seasons, optimizing parameters to ensure profitability in each market regime.

## Political Seasons Analyzed:
- **2012-2016**: Obama second term
- **2016-2021**: Trump administration + COVID-19
- **2021-2024**: Biden administration
- **2024-2026**: Post-2024 election period

## Objectives:
1. Optimize strategy parameters for profitability in each political season
2. Improve backtester robustness to mirror QuantConnect's Lean architecture
3. Generate comprehensive results, visualizations, and parameter configurations
4. Compare performance across different market regimes

## Strategy Overview:
- Intraday momentum breakouts in ES/NQ futures
- Noise area breakout detection with confirmation bars
- Volatility-based position sizing
- Multiple exit conditions (time, momentum failure, profit targets)

## 1. Data Loading and Period Division

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import yaml
import json
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# Add current directory to path for imports
sys.path.append('.')

# Import our strategy components
from noise_area import NoiseAreaCalculator
from signal_generator import SignalGenerator
from position_sizer import PositionSizer
from enhanced_backtester import EnhancedBacktester, OrderSide, OrderType

# Set up plotting and warnings
plt.style.use('default')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

In [ ]:
# Load market data
print("Loading ES and NQ market data...")

es_data = pd.read_csv(
    'Data/ES_5min_RTH.csv',
    index_col='ts_event',
    parse_dates=True
)

nq_data = pd.read_csv(
    'Data/NQ_5min_RTH.csv',
    index_col='ts_event',
    parse_dates=True
)

# Ensure timezone-aware DatetimeIndex
es_data.index = pd.to_datetime(es_data.index, utc=True).tz_convert('US/Eastern')
nq_data.index = pd.to_datetime(nq_data.index, utc=True).tz_convert('US/Eastern')

print(f"ES data: {len(es_data):,} bars from {es_data.index[0]} to {es_data.index[-1]}")
print(f"NQ data: {len(nq_data):,} bars from {nq_data.index[0]} to {nq_data.index[-1]}")

# Define political seasons
political_seasons = {
    '2012_2016': {
        'name': 'Obama Term 2',
        'start': '2012-01-01',
        'end': '2016-12-31',
        'data': {}
    },
    '2016_2021': {
        'name': 'Trump + COVID',
        'start': '2016-01-01',
        'end': '2021-12-31',
        'data': {}
    },
    '2021_2024': {
        'name': 'Biden Term',
        'start': '2021-01-01',
        'end': '2024-12-31',
        'data': {}
    },
    '2024_2026': {
        'name': 'Post-2024',
        'start': '2024-01-01',
        'end': '2026-12-31',
        'data': {}
    }
}

# Filter data for each season
for season_key, season_info in political_seasons.items():
    print(f"\nProcessing {season_info['name']} ({season_key})...")

    # Convert string dates to timezone-aware Timestamps to match data timezone
    start_date = pd.Timestamp(season_info['start'], tz='US/Eastern')
    end_date = pd.Timestamp(season_info['end'], tz='US/Eastern')

    # Filter ES data
    es_mask = (es_data.index >= start_date) & (es_data.index <= end_date)
    es_season = es_data[es_mask].copy()

    # Filter NQ data
    nq_mask = (nq_data.index >= start_date) & (nq_data.index <= end_date)
    nq_season = nq_data[nq_mask].copy()

    # Store filtered data
    season_info['data'] = {
        'ES': es_season,
        'NQ': nq_season
    }

    print(f"  ES: {len(es_season):,} bars")
    print(f"  NQ: {len(nq_season):,} bars")

print("\nData loading and period division complete!")

## 2. Enhanced Backtester Architecture

The enhanced backtester includes Lean/QuantConnect-style features:
- Event-driven order management
- Realistic slippage and market impact modeling
- Risk management with position limits and margin requirements
- Portfolio-level position tracking
- Comprehensive performance analytics

## 3. Parameter Optimization Setup

Define parameter ranges for optimization across different market regimes. We'll optimize for Sharpe ratio while ensuring positive returns.

In [ ]:
# Load base configuration
with open('config.yaml', 'r') as f:
    base_config = yaml.safe_load(f)

# Define parameter optimization ranges
param_ranges = {
    'noise_area_lookback_days': [10, 20, 30, 60],
    'target_daily_volatility': [0.05, 0.10, 0.15, 0.20, 0.25],
    'atr_multiplier': [1.0, 1.5, 2.0, 2.5],
    'confirmation_bars': [1, 2, 3],
    'volume_threshold_percentile': [50, 60, 70, 80],
    'min_signal_strength': [30, 40, 50, 60],
    'max_hold_bars': [39, 78, 117],  # 1, 2, 3 trading days
    'trailing_stop_percent': [0.005, 0.01, 0.015, 0.02],
    'slippage_ticks': [0.25, 0.5, 1.0],
    'commission_per_contract': [1.0, 2.0, 4.2]
}

print("Parameter ranges defined:")
for param, values in param_ranges.items():
    print(f"  {param}: {values}")

# Optimization settings
N_OPTIMIZATION_TRIALS = 30  # Bayesian optimization trials (much faster than grid search)
OPTIMIZATION_METRIC = 'sharpe_ratio'
MIN_SHARPE_THRESHOLD = 0.5  # Minimum acceptable Sharpe ratio

# Results storage
optimization_results = {}
season_results = {}

print(f"\nOptimization settings:")
print(f"  Number of trials: {N_OPTIMIZATION_TRIALS}")
print(f"  Optimization metric: {OPTIMIZATION_METRIC}")
print(f"  Min Sharpe threshold: {MIN_SHARPE_THRESHOLD}")

In [ ]:
def create_config_variant(base_config: dict, param_dict: dict) -> dict:
    """Create configuration variant with parameter changes."""
    config = base_config.copy()

    # Update strategy parameters
    config['strategy']['noise_area']['lookback_days'] = param_dict['noise_area_lookback_days']
    config['strategy']['position_sizing']['target_daily_volatility'] = param_dict['target_daily_volatility']
    config['strategy']['noise_area']['atr_multiplier'] = param_dict['atr_multiplier']
    config['strategy']['entry_exit']['confirmation_bars'] = param_dict['confirmation_bars']
    config['strategy']['entry_exit']['volume_threshold_percentile'] = param_dict['volume_threshold_percentile']
    config['strategy']['entry_exit']['min_signal_strength'] = param_dict['min_signal_strength']
    config['strategy']['exit']['max_hold_bars'] = param_dict['max_hold_bars']
    config['strategy']['exit']['trailing_stop_percent'] = param_dict['trailing_stop_percent']

    # Update transaction costs
    config['strategy']['transaction_costs']['slippage_ticks'] = param_dict['slippage_ticks']
    config['strategy']['transaction_costs']['commission_per_contract'] = param_dict['commission_per_contract']

    return config

def generate_signals_for_config(config: dict, es_data: pd.DataFrame, nq_data: pd.DataFrame) -> dict:
    """Generate trading signals using current config."""
    try:
        # Initialize components
        calculator = NoiseAreaCalculator(config)
        signal_gen = SignalGenerator(config)
        sizer = PositionSizer(config)

        # Process ES data
        es_processed = es_data.copy()
        es_processed = calculator.calculate_noise_area(es_processed)
        es_processed = calculator.identify_breakouts(es_processed)
        es_processed = signal_gen.generate_signals(es_processed)

        # Process NQ data
        nq_processed = nq_data.copy()
        nq_processed = calculator.calculate_noise_area(nq_processed)
        nq_processed = calculator.identify_breakouts(nq_processed)
        nq_processed = signal_gen.generate_signals(nq_processed)

        # Position sizing
        portfolio = sizer.calculate_portfolio_positions(es_processed, nq_processed)

        return portfolio

    except Exception as e:
        print(f"Error generating signals: {str(e)}")
        return {}

def optimize_season_parameters(season_key: str, season_data: dict, param_ranges: dict,
                             n_trials: int = 50) -> dict:
    """Optimize parameters for a specific season using Bayesian optimization."""
    print(f"\n{'='*60}")
    print(f"OPTIMIZING PARAMETERS FOR {season_key.upper()}")
    print(f"{'='*60}")

    def objective(trial):
        """Objective function for Optuna optimization."""
        # Sample parameters from the search space
        param_dict = {}
        for param_name, values in param_ranges.items():
            if param_name in ['noise_area_lookback_days', 'confirmation_bars', 'volume_threshold_percentile',
                            'min_signal_strength', 'max_hold_bars']:
                # Integer parameters
                param_dict[param_name] = trial.suggest_int(param_name, min(values), max(values))
            else:
                # Float parameters
                param_dict[param_name] = trial.suggest_float(param_name, min(values), max(values))

        try:
            # Create config variant
            config = create_config_variant(base_config, param_dict)

            # Generate signals
            signals_data = generate_signals_for_config(config, season_data['ES'], season_data['NQ'])

            if not signals_data:
                return -np.inf  # Penalize failed signal generation

            # Run backtest
            backtester = EnhancedBacktester(config)
            backtester.load_market_data({'ES': signals_data.get('ES_momentum', pd.DataFrame()),
                                       'NQ': signals_data.get('NQ_momentum', pd.DataFrame())})

            equity_curve = backtester.run_backtest(signals_data)

            # Get Sharpe ratio (our optimization target)
            sharpe = backtester.performance_metrics.get('sharpe_ratio', -np.inf)

            # Also consider total return as secondary objective
            total_return = backtester.performance_metrics.get('total_return', -np.inf)

            # Combine metrics: prioritize Sharpe but also consider returns
            combined_score = sharpe + (total_return * 0.1)  # Weight returns less

            return combined_score if not np.isinf(combined_score) else -np.inf

        except Exception as e:
            # Penalize failed backtests
            return -np.inf

    # Create Optuna study
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42),  # Reproducible results
        pruner=MedianPruner()
    )

    print(f"Running Bayesian optimization with {n_trials} trials...")

    # Run optimization
    study.optimize(objective, n_trials=n_trials, timeout=3600)  # 1 hour timeout

    # Get best parameters
    best_trial = study.best_trial
    best_params = best_trial.params
    best_score = best_trial.value

    print(f"Best trial score: {best_score:.3f}")
    print("Best parameters:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")

    # Run final backtest with best parameters to get full results
    try:
        config = create_config_variant(base_config, best_params)
        signals_data = generate_signals_for_config(config, season_data['ES'], season_data['NQ'])

        if signals_data:
            backtester = EnhancedBacktester(config)
            backtester.load_market_data({'ES': signals_data.get('ES_momentum', pd.DataFrame()),
                                       'NQ': signals_data.get('NQ_momentum', pd.DataFrame())})

            equity_curve = backtester.run_backtest(signals_data)

            best_results = {
                'equity_curve': equity_curve,
                'performance_metrics': backtester.performance_metrics,
                'trades': backtester.get_trades_dataframe(),
                'config': config
            }
        else:
            best_results = None

    except Exception as e:
        print(f"Final backtest failed: {e}")
        best_results = None

    return {
        'best_params': best_params,
        'best_results': best_results,
        'best_score': best_score,
        'study': study  # Keep study for analysis
    }

print("Optimization functions defined successfully")

## 4. Backtesting 2012-2016 Period (Obama Term 2)

This period represents a relatively stable market environment with moderate volatility and steady growth.

In [ ]:
# Optimize parameters for 2012-2016
season_2012_2016 = '2012_2016'
season_data = political_seasons[season_2012_2016]['data']

print(f"Starting optimization for {political_seasons[season_2012_2016]['name']}")
print(f"Data range: {season_data['ES'].index[0]} to {season_data['ES'].index[-1]}")
print(f"ES bars: {len(season_data['ES']):,}")
print(f"NQ bars: {len(season_data['NQ']):,}")

# Run optimization
optimization_results[season_2012_2016] = optimize_season_parameters(
    season_2012_2016, season_data, param_ranges, N_OPTIMIZATION_TRIALS
)

# Store results
if optimization_results[season_2012_2016]['best_params'] is not None:
    season_results[season_2012_2016] = optimization_results[season_2012_2016]['best_results']

    print(f"\nBest parameters for {season_2012_2016}:")
    for param, value in optimization_results[season_2012_2016]['best_params'].items():
        print(f"  {param}: {value}")

    print(f"\nPerformance metrics:")
    metrics = optimization_results[season_2012_2016]['best_results']['performance_metrics']
    print(f"  Total Return: {metrics['total_return']:.2%}")
    print(f"  Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
    print(f"  Max Drawdown: {metrics['max_drawdown']:.2%}")
    print(f"  Win Rate: {metrics['win_rate']:.1%}")
    print(f"  Total Trades: {metrics['total_trades']}")
else:
    print(f"Optimization failed for {season_2012_2016}")

## 5. Backtesting 2016-2021 Period (Trump + COVID)

This period includes significant market volatility from the COVID-19 crash and recovery.

In [ ]:
# Optimize parameters for 2016-2021
season_2016_2021 = '2016_2021'
season_data = political_seasons[season_2016_2021]['data']

print(f"Starting optimization for {political_seasons[season_2016_2021]['name']}")
print(f"Data range: {season_data['ES'].index[0]} to {season_data['ES'].index[-1]}")

optimization_results[season_2016_2021] = optimize_season_parameters(
    season_2016_2021, season_data, param_ranges, N_OPTIMIZATION_TRIALS
)

if optimization_results[season_2016_2021]['best_params'] is not None:
    season_results[season_2016_2021] = optimization_results[season_2016_2021]['best_results']
    print(f"Optimization completed for {season_2016_2021}")
else:
    print(f"Optimization failed for {season_2016_2021}")

## 6. Backtesting 2021-2024 Period (Biden Term)

This period represents post-COVID recovery with inflationary pressures.

In [ ]:
# Optimize parameters for 2021-2024
season_2021_2024 = '2021_2024'
season_data = political_seasons[season_2021_2024]['data']

print(f"Starting optimization for {political_seasons[season_2021_2024]['name']}")
print(f"Data range: {season_data['ES'].index[0]} to {season_data['ES'].index[-1]}")

optimization_results[season_2021_2024] = optimize_season_parameters(
    season_2021_2024, season_data, param_ranges, N_OPTIMIZATION_TRIALS
)

if optimization_results[season_2021_2024]['best_params'] is not None:
    season_results[season_2021_2024] = optimization_results[season_2021_2024]['best_results']
    print(f"Optimization completed for {season_2021_2024}")
else:
    print(f"Optimization failed for {season_2021_2024}")

## 7. Backtesting 2024-2026 Period (Post-2024)

This period represents the most recent market data with high volatility and uncertainty.

In [ ]:
# Optimize parameters for 2024-2026
season_2024_2026 = '2024_2026'
season_data = political_seasons[season_2024_2026]['data']

print(f"Starting optimization for {political_seasons[season_2024_2026]['name']}")
print(f"Data range: {season_data['ES'].index[0]} to {season_data['ES'].index[-1]}")

optimization_results[season_2024_2026] = optimize_season_parameters(
    season_2024_2026, season_data, param_ranges, N_OPTIMIZATION_TRIALS
)

if optimization_results[season_2024_2026]['best_params'] is not None:
    season_results[season_2024_2026] = optimization_results[season_2024_2026]['best_results']
    print(f"Optimization completed for {season_2024_2026}")
else:
    print(f"Optimization failed for {season_2024_2026}")

## 8. Results Analysis and Visualization

Compare performance across all political seasons and analyze parameter evolution.

In [ ]:
# Create comprehensive results analysis
print("Generating comprehensive results analysis...")

# Collect metrics across seasons
season_metrics = []
equity_curves = {}
season_params = {}

for season_key, results in season_results.items():
    metrics = results['performance_metrics'].copy()
    metrics['season'] = season_key
    metrics['season_name'] = political_seasons[season_key]['name']
    season_metrics.append(metrics)

    # Normalize equity curves for comparison
    equity = results['equity_curve'].copy()
    initial_value = equity['portfolio_value'].iloc[0]
    equity['normalized_value'] = equity['portfolio_value'] / initial_value
    equity_curves[season_key] = equity

    # Store parameters
    season_params[season_key] = optimization_results[season_key]['best_params']

# Create metrics DataFrame
metrics_df = pd.DataFrame(season_metrics)

# Display summary table
print("\n" + "="*80)
print("POLITICAL SEASON PERFORMANCE SUMMARY")
print("="*80)
summary_cols = ['season_name', 'total_return', 'sharpe_ratio', 'max_drawdown', 'win_rate', 'total_trades']
print(metrics_df[summary_cols].to_string(index=False, float_format='%.2f'))
print("="*80)

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Political Season Strategy Performance Comparison', fontsize=16)

# 1. Sharpe Ratio Comparison
axes[0, 0].bar(metrics_df['season_name'], metrics_df['sharpe_ratio'])
axes[0, 0].set_title('Sharpe Ratio by Season')
axes[0, 0].set_ylabel('Sharpe Ratio')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Total Return Comparison
axes[0, 1].bar(metrics_df['season_name'], metrics_df['total_return'])
axes[0, 1].set_title('Total Return by Season')
axes[0, 1].set_ylabel('Total Return (%)')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Win Rate Comparison
axes[1, 0].bar(metrics_df['season_name'], metrics_df['win_rate'])
axes[1, 0].set_title('Win Rate by Season')
axes[1, 0].set_ylabel('Win Rate')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Max Drawdown Comparison
axes[1, 1].bar(metrics_df['season_name'], -metrics_df['max_drawdown'])  # Negative for visualization
axes[1, 1].set_title('Max Drawdown by Season (Absolute)')
axes[1, 1].set_ylabel('Max Drawdown (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('results/political_season_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Equity curves comparison
plt.figure(figsize=(12, 8))

for season_key, equity in equity_curves.items():
    season_name = political_seasons[season_key]['name']
    plt.plot(equity.index, equity['normalized_value'], label=season_name, linewidth=2)

plt.title('Normalized Equity Curves by Political Season')
plt.xlabel('Date')
plt.ylabel('Portfolio Value (Normalized)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('results/political_season_equity_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# Parameter evolution heatmap
param_data = []
for season_key, params in season_params.items():
    row = {'season': political_seasons[season_key]['name']}
    row.update(params)
    param_data.append(row)

if param_data:
    params_df = pd.DataFrame(param_data)
    params_df.set_index('season', inplace=True)

    plt.figure(figsize=(12, 8))
    sns.heatmap(params_df.T, annot=True, cmap='YlOrRd', fmt='.2f')
    plt.title('Optimal Parameters by Political Season')
    plt.tight_layout()
    plt.savefig('results/political_season_parameter_evolution.png', dpi=300, bbox_inches='tight')
    plt.show()

print("Visualization complete - saved to results/ directory")

## 9. Saving Results and Parameters

Save all optimized parameters, performance metrics, equity curves, and visualizations for each political season.

In [ ]:
# Create output directory
output_dir = Path("results/political_season_analysis")
output_dir.mkdir(parents=True, exist_ok=True)

print("Saving comprehensive results...")

# Save season comparison metrics
metrics_df.to_csv(output_dir / 'season_comparison_metrics.csv', index=False)

# Save best parameters summary
params_summary = {}
for season_key, params in season_params.items():
    params_summary[season_key] = {
        'season_name': political_seasons[season_key]['name'],
        'parameters': params,
        'performance': season_results[season_key]['performance_metrics']
    }

with open(output_dir / 'best_parameters_summary.yaml', 'w') as f:
    yaml.dump(params_summary, f, default_flow_style=False)

# Save individual season results
for season_key, results in season_results.items():
    season_dir = output_dir / season_key
    season_dir.mkdir(exist_ok=True)

    # Save equity curve
    results['equity_curve'].to_csv(season_dir / 'equity_curve.csv')

    # Save trades
    results['trades'].to_csv(season_dir / 'trades.csv', index=False)

    # Save performance metrics
    with open(season_dir / 'performance_metrics.json', 'w') as f:
        json.dump(results['performance_metrics'], f, indent=2, default=str)

    # Save optimized parameters
    with open(season_dir / 'optimal_parameters.yaml', 'w') as f:
        yaml.dump(optimization_results[season_key]['best_params'], f, default_flow_style=False)

    # Save config
    with open(season_dir / 'config_used.yaml', 'w') as f:
        yaml.dump(results['config'], f, default_flow_style=False)

# Save optimization metadata
metadata = {
    'optimization_settings': {
        'max_combinations': MAX_COMBINATIONS,
        'optimization_metric': OPTIMIZATION_METRIC,
        'min_sharpe_threshold': MIN_SHARPE_THRESHOLD,
        'param_ranges': param_ranges
    },
    'seasons_analyzed': list(season_results.keys()),
    'timestamp': datetime.now().isoformat(),
    'total_combinations_tested': len(season_results) * MAX_COMBINATIONS
}

with open(output_dir / 'optimization_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"\nAll results saved to: {output_dir}")
print("\nFiles saved:")
print("- season_comparison_metrics.csv")
print("- best_parameters_summary.yaml")
print("- political_season_metrics_comparison.png")
print("- political_season_equity_curves.png")
print("- political_season_parameter_evolution.png")
print("- Individual season folders with detailed results")

# Print final summary
print("\n" + "="*80)
print("POLITICAL SEASON ANALYSIS COMPLETE")
print("="*80)
print("Summary of Results:")
for season_key in season_results.keys():
    season_name = political_seasons[season_key]['name']
    metrics = season_results[season_key]['performance_metrics']
    params = season_params[season_key]

    print(f"\n{season_name} ({season_key}):")
    print(".2f")
    print(".2f")
    print(".1%")
    print(f"  Key Parameters: volatility={params['target_daily_volatility']}, "
          f"lookback={params['noise_area_lookback_days']}, "
          f"confirmation={params['confirmation_bars']}")

print(f"\nDetailed results available in: {output_dir}")
print("="*80)